In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6,7"
os.environ["HF_HOME"] = "/local1/mohsenfayyaz/.hfcache/"

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("reasonir/ReasonIR-8B", torch_dtype="auto", trust_remote_code=True)
model = model.to("cuda")
model.eval()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

ReasonIRModel(
  (embed_tokens): Embedding(128256, 4096)
  (layers): ModuleList(
    (0-31): 32 x LlamaDecoderLayer(
      (self_attn): LlamaSdpaAttention(
        (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        (rotary_emb): LlamaRotaryEmbedding()
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
        (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
        (act_fn): SiLU()
      )
      (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((4096,), eps=1e-05)
  (rotary_emb): LlamaRotaryEm

In [4]:
def get_model_scores(query, document):
    query_instruction = ""
    doc_instruction = ""
    query_emb = model.encode(query, instruction=query_instruction)
    doc_emb = model.encode(document, instruction=doc_instruction)
    sim = query_emb @ doc_emb.T
    return sim

In [11]:
from tqdm.auto import tqdm
import pandas as pd

# Login using e.g. `huggingface-cli login` to access this dataset
df_foil = pd.read_json("hf://datasets/mohsenfayyaz/ColDeR/test/foil.jsonl", lines=True)
df_foil.head(1)

rdf = []
for row in tqdm(df_foil.to_dict(orient="records")):
    query = row["query"]
    sim_1 = get_model_scores(query, row["document_1"])
    sim_2 = get_model_scores(query, row["document_2"])
    rdf.append({"query": query, "document_1": row["document_1"], "document_2": row["document_2"], "score_1": sim_1, "score_2": sim_2})
    
rdf = pd.DataFrame(rdf)
rdf

  0%|          | 0/250 [00:00<?, ?it/s]

,query,document_1,document_2,score_1,score_2
0,When was The Private Life of Helen of Troy pub...,""" The Private Life of Helen of Troy "" "" The Pr...",The discography of English rock band Joy Divis...,0.446776,0.317639
1,What is EMH an instance of?,""" EMH "" "" EMH "" Guest star Leland Orser plays ...",The University of Uyo ( UNIUYO ) is located in...,0.190340,0.150067
2,Which record label is God 's Son associated with?,""" God 's Son "" "" God 's Son "" Partly inspired ...","The Metacomet Ridge , Metacomet Ridge Mountain...",0.543173,0.304694
3,What is a notable work of Miami Sound Machine?,""" Miami Sound Machine "" "" Miami Sound Machine ...",Yuriy Vitaliyovych Lutsenko (; born 14 Decembe...,0.508363,0.282049
4,Which country is Lake Ewauna associated with?,""" Lake Ewauna "" "" Lake Ewauna "" The Klamath Ri...","Henry Wager Halleck ( January 16 , 1815 – Janu...",0.626632,0.343271
...,...,...,...,...,...
245,Which country is Bad Astronaut associated with?,""" Bad Astronaut "" "" Bad Astronaut "" In Bad Ast...",Robert Kingsbury Huntington ( 13 March 1921 – ...,0.434960,0.323502
246,What conflict was Catinat part of?,""" Catinat "" "" Catinat "" Marshal Villeroi repla...","In gridiron football , a triple - threat man i...",0.537630,0.243081
247,Who is the father of Billy?,""" Billy "" "" Billy "" Corral on October 26 , 188...","Tire ( ) is a populous district , as well as t...",0.300026,0.106065
248,Which administrative territorial entity is Dur...,""" Durgada "" "" Durgada "" Durgada has a railway ...",The Denali National Park Improvement Act ( ) i...,0.482737,0.305889


In [12]:
import numpy as np
from scipy import stats

def standard_ttest_ppf(n, confidence_level=0.95):
    return stats.t.ppf(q=1-confidence_level, df=n-1, loc=0, scale=1)

df = rdf.copy()
col1, col2 = "score_2", "score_1"
ttest = stats.ttest_rel(df[col1], df[col2])
result = {
    "col1": col1,
    "col2": col2,
    "ttest_stats": ttest[0],
    "ttest_pvalue": ttest[1],
    "ttest_ci_low_stats": ttest.confidence_interval(confidence_level=0.95)[0],
    "ttest_ci_high_stats": ttest.confidence_interval(confidence_level=0.95)[1],
    "ttest_ci_low": np.abs(standard_ttest_ppf(len(df))),
    "ttest_ci_high": np.abs(standard_ttest_ppf(len(df))),
    "standard_ttest_ppf": standard_ttest_ppf(len(df)),
    "acc": (df[col1] > df[col2]).mean(),
}
result

{'col1': 'score_2',
 'col2': 'score_1',
 'ttest_stats': -36.922937203689706,
 'ttest_pvalue': 5.485491239398149e-103,
 'ttest_ci_low_stats': -0.23730227908934348,
 'ttest_ci_high_stats': -0.2132680360332609,
 'ttest_ci_low': 1.650996151677261,
 'ttest_ci_high': 1.650996151677261,
 'standard_ttest_ppf': -1.650996151677261,
 'acc': 0.008}

# RUN ALL

In [7]:
import pandas as pd
from tqdm.auto import tqdm
from scipy import stats


def standard_ttest_ppf(n, confidence_level=0.95):
    return stats.t.ppf(q=1-confidence_level, df=n-1, loc=0, scale=1)

results = []
for part in tqdm(["brevity_bias", "answer_importance", "repetition_bias", "position_bias", "literal_bias"]):
    df_foil = pd.read_json(f"hf://datasets/mohsenfayyaz/ColDeR/test/{part}.jsonl", lines=True)

    rdf = []
    for row in tqdm(df_foil.to_dict(orient="records")):
        query = row["query"]
        sim_1 = get_model_scores(query, row["document_1"])
        sim_2 = get_model_scores(query, row["document_2"])
        rdf.append({"query": query, "document_1": row["document_1"], "document_2": row["document_2"], "score_1": sim_1, "score_2": sim_2})
        
    rdf = pd.DataFrame(rdf)
    
    df = rdf.copy()
    col1, col2 = "score_1", "score_2"
    ttest = stats.ttest_rel(df[col1], df[col2])
    result = {
        "col1": col1,
        "col2": col2,
        "ttest_stats": ttest[0],
        "ttest_pvalue": ttest[1],
        "ttest_ci_low_stats": ttest.confidence_interval(confidence_level=0.95)[0],
        "ttest_ci_high_stats": ttest.confidence_interval(confidence_level=0.95)[1],
        "ttest_ci_low": np.abs(standard_ttest_ppf(len(df))),
        "ttest_ci_high": np.abs(standard_ttest_ppf(len(df))),
        "standard_ttest_ppf": standard_ttest_ppf(len(df)),
        "acc": (df[col1] > df[col2]).mean(),
    }
    results.append({
        'Model': "ReasonIR-8B", 
        'col1': "doc1", 
        'col2': "doc2",
        'Paired t-Test Statistic': result['ttest_stats'], 
        'ttest_pvalue': result['ttest_pvalue'],
        'ttest_ci_low_stats': result['ttest_ci_low_stats'],
        'ttest_ci_high_stats': result['ttest_ci_high_stats'],
        'ttest_ci_low': result['ttest_ci_low'],
        'ttest_ci_high': result['ttest_ci_high'],
        'standard_ttest_ppf': result['standard_ttest_ppf'],
        'acc': result['acc'],
        'mean_diff': "NAN",
        'std_diff': "NAN",
        'n': len(df),
        'test': part,
    })

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

  0%|          | 0/250 [00:00<?, ?it/s]

In [8]:
print(results)

[{'Model': 'ReasonIR-8B', 'col1': 'doc1', 'col2': 'doc2', 'Paired t-Test Statistic': 32.78105115905685, 'ttest_pvalue': 2.608983675013358e-92, 'ttest_ci_low_stats': 0.1534190782993777, 'ttest_ci_high_stats': 0.17303281932177708, 'ttest_ci_low': 1.650996151677261, 'ttest_ci_high': 1.650996151677261, 'standard_ttest_ppf': -1.650996151677261, 'acc': 0.996, 'mean_diff': 'NAN', 'std_diff': 'NAN', 'n': 250, 'test': 'brevity_bias'}, {'Model': 'ReasonIR-8B', 'col1': 'doc1', 'col2': 'doc2', 'Paired t-Test Statistic': 10.533721367939853, 'ttest_pvalue': 1.0676319675083417e-21, 'ttest_ci_low_stats': 0.028912666784240704, 'ttest_ci_high_stats': 0.0422109704747658, 'ttest_ci_low': 1.650996151677261, 'ttest_ci_high': 1.650996151677261, 'standard_ttest_ppf': -1.650996151677261, 'acc': 0.792, 'mean_diff': 'NAN', 'std_diff': 'NAN', 'n': 250, 'test': 'answer_importance'}, {'Model': 'ReasonIR-8B', 'col1': 'doc1', 'col2': 'doc2', 'Paired t-Test Statistic': 5.917222674199121, 'ttest_pvalue': 1.077768198606